# Better Brush — Fine-tune SAM 2.1 for anime / AMV on Google Colab

This notebook trains a **SAM 2.1 Base+** checkpoint on your own corrected
masks so the model learns *your* anime / AMV look (flat cel shading, crisp line
art, hair strands) instead of being told to accept messier candidates.

**Before you start**
1. In Better Brush: matte a clip, correct the masks, then
   **File → Save Training Data…** and repeat over 5–20 clips.
2. Put that `dataset/` folder in Google Drive (or zip it and upload it).
3. `Runtime → Change runtime type → GPU` (free **T4** is enough; **L4/A100**
   on Colab Pro is faster and gets bfloat16 automatically).

**You will need** the two helper files that ship next to this notebook
(`tools/sam2_training/colab/`):
* `anime_amv_colab.yaml` — the tuned recipe
* `sam2_colab_finetune.py` — clone / install / configure / train driver


## 1 · Check the GPU

In [ ]:
!nvidia-smi -L
import torch
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available())
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    print("compute capability", f"{major}.{minor}",
          "-> bfloat16 OK" if major >= 8 else "-> will use float16")


## 2 · Get the dataset and the helper files into Colab

Mount Drive (recommended, so your dataset and the trained checkpoint survive a
disconnect). Skip the mount and just upload a zip if you prefer.

Expected dataset layout (exactly what Better Brush exports):

```
dataset/
  JPEGImages/<clip>/00042.jpg
  Annotations/<clip>/0/00042.png
  ImageSets/train.txt
```


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# Point this at the dataset folder on Drive. If you uploaded a zip
# instead, unzip it somewhere and point DATASET there.
DATASET = "/content/drive/MyDrive/BetterBrush/dataset"

import os
print("train.txt present:", os.path.isfile(f"{DATASET}/ImageSets/train.txt"))


In [ ]:
# Upload anime_amv_colab.yaml and sam2_colab_finetune.py if they are not
# already in this session (they live in tools/sam2_training/colab/).
from pathlib import Path
missing = [f for f in ("anime_amv_colab.yaml", "sam2_colab_finetune.py")
           if not Path(f).is_file()]
if missing:
    from google.colab import files
    print("Please upload:", ", ".join(missing))
    files.upload()
else:
    print("helper files already present")


## 3 · Prepare (clone SAM 2, install, download checkpoint, write config)

Auto-detects the GPU and picks `bfloat16` on L4/A100 or `float16` on the free
T4. On the T4 keep `--resolution 384 --num-frames 2`; on L4/A100 you can try
`--resolution 512 --num-frames 4`.


In [ ]:
!python sam2_colab_finetune.py --dataset "$DATASET" \
    --resolution 384 --num-frames 2 --no-train


## 4 · Train

In [ ]:
%cd /content/sam2
!python training/train.py \
    -c configs/sam2.1_training/anime_amv_colab.yaml \
    --use-cluster 0 --num-gpus 1


### Watch the loss (optional, second tab)

Open a terminal from the Colab sidebar (or a new cell) and run:

```bash
tensorboard --logdir /content/sam2/sam2_logs --port 6006
```

`loss_mask` should fall then flatten. 20 epochs is usually plenty on a small
dataset; more mostly memorises your clips.


## 5 · Save the trained checkpoint back to Drive

In [ ]:
import glob, os, shutil, time
ckpts = sorted(glob.glob("/content/sam2/sam2_logs/**/*.pt", recursive=True),
               key=os.path.getmtime, reverse=True)
for c in ckpts[:5]:
    print(time.strftime("%H:%M:%S", time.localtime(os.path.getmtime(c))), c)

out_dir = "/content/drive/MyDrive/BetterBrush/sam2_checkpoints"
os.makedirs(out_dir, exist_ok=True)
if ckpts:
    dst = os.path.join(out_dir, "anime_amv_v1.pt")
    shutil.copy2(ckpts[0], dst)
    print("saved ->", dst)
else:
    print("no checkpoint found yet — is training still running?")


## 6 · Use it in Better Brush

1. Copy `anime_amv_v1.pt` into `<Better-Brush>/checkpoints/custom/`.
2. Create `<Better-Brush>/checkpoints/custom/sam2_custom.json`:

```json
{
  "checkpoint": "anime_amv_v1.pt",
  "config": "sam2.1_hiera_b+.yaml",
  "label": "Anime AMV v1"
}
```

3. Restart Better Brush and pick **Anime AMV v1** in the **AI Model** dropdown
   (or per shot in **Inspect → Shot Profiles**).

Keep the stock Base+ checkpoint around — rename the sidecar to switch back.
